# 古籍批量自动标点 — Colab A100 + n_parallel=4

基于 llama.cpp 原生 server + 4 路并发 HTTP 请求。Qwen3-30B-A3B Q4_K_M。

**Runtime（务必）**：Runtime → Change runtime type → **A100 GPU** + Premium。
脚本会强制校验显存 ≥ 40GB，T4/L4/V100 会立即报错，不会浪费配额。

## 0. 配置

In [13]:
import os

# Kaggle 凭据（这本 notebook 不入版本库，写死省手动）
os.environ['KAGGLE_USERNAME'] = 'canhuilipku'
os.environ['KAGGLE_KEY']      = '180d316a582936af82f3122d1378d868'

BOOK = 'mingshilu-p4-prechunked'   # P5 预切版：长段已本地切到 ≤350 字带 marker
PART = 1
PART_OF = 1

N_PARALLEL = 4
MODEL_REPO = 'bartowski/Qwen_Qwen3-30B-A3B-Instruct-2507-GGUF'
MODEL_PATTERN = 'Q4_K_M'

# 注意：Colab 上 8080 经常被别的服务占，换个冷门端口
SERVER_PORT = 18765

# 跑完后回传输出用的 dataset slug（不存在自动创建）
OUTPUT_DATASET = 'canhuilipku/mingshilu-p4-output'

SUFFIX = f'-p{PART}of{PART_OF}' if PART_OF > 1 else ''
print(f'book={BOOK} part={PART}/{PART_OF} parallel={N_PARALLEL} model={MODEL_PATTERN}')
print(f'输出回传到 https://www.kaggle.com/datasets/{OUTPUT_DATASET}')


book=mingshilu-p4-prechunked part=1/1 parallel=4 model=Q4_K_M
输出回传到 https://www.kaggle.com/datasets/canhuilipku/mingshilu-p4-output


## 1. GPU 检查

In [14]:
import subprocess
info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader']).decode().strip()
print(info)
gpus = info.splitlines()
print(f'\n{len(gpus)} GPU(s) detected')

# 硬断言：必须 A100 / H100（显存 ≥ 40GB）。把 Q4_K_M (~17GB) + N_PARALLEL=4 × 4096 ctx 的 KV cache 塞进
# L4 24GB 会很险；T4 16GB 一定 OOM。这里 fail fast，省得跑到一半挂。
name, mem_str, _cc = [s.strip() for s in gpus[0].split(',')]
mem_mib = int(mem_str.replace(' MiB', ''))
assert mem_mib >= 40000, (
    f'❌ 当前 GPU {name} 显存 {mem_mib} MiB，不满足 ≥ 40GB（A100/H100）。'
    f'去 Runtime → Change runtime type 选 A100 GPU 后再跑。'
)
print(f'\n✅ GPU 校验通过：{name} ({mem_mib} MiB)')

NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0

1 GPU(s) detected

✅ GPU 校验通过：NVIDIA A100-SXM4-80GB (81920 MiB)


## 2. 装 Python 依赖

In [15]:
%pip install -q opencc-python-reimplemented yitizi httpx kagglehub huggingface_hub

## 3. 编译 llama.cpp（含 CUDA），~3-5 min

In [16]:
import os, subprocess, time

LLAMA_DIR = '/tmp/llama.cpp'
SERVER_BIN = f'{LLAMA_DIR}/build/bin/llama-server'

if not os.path.exists(SERVER_BIN):
    t0 = time.time()
    print('clone llama.cpp...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp', LLAMA_DIR], check=True)
    print('cmake configure (CUDA on)...')
    subprocess.run(['cmake', '-B', f'{LLAMA_DIR}/build',
                    '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release',
                    '-DLLAMA_CURL=OFF'],
                   cwd=LLAMA_DIR, check=True)
    print('build ...')
    subprocess.run(['cmake', '--build', f'{LLAMA_DIR}/build',
                    '--config', 'Release', '--target', 'llama-server',
                    '-j', '8'], cwd=LLAMA_DIR, check=True)
    print(f'\nbuild done in {(time.time()-t0)/60:.1f} min')
else:
    print('llama-server already built')

assert os.path.exists(SERVER_BIN), f'{SERVER_BIN} not found'
print(f'binary: {SERVER_BIN}')

llama-server already built
binary: /tmp/llama.cpp/build/bin/llama-server


## 4. 下载 GGUF 模型（Q4_K_M，~10 GB，~2-4 min）

In [17]:
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache/hub'

from huggingface_hub import HfApi, hf_hub_download
api = HfApi()
files = api.list_repo_files(MODEL_REPO)
matches = sorted([f for f in files if MODEL_PATTERN in f and f.endswith('.gguf')
                  and '_L' not in f and 'imatrix' not in f])
print(f'{MODEL_PATTERN} files in repo:')
for m in matches: print('  -', m)
assert matches, 'no matching gguf'

MODEL_FILE = matches[0]
print(f'\ndownloading {MODEL_FILE}...')
t0 = time.time()
MODEL_PATH = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(f'done in {(time.time()-t0)/60:.1f} min, size {os.path.getsize(MODEL_PATH)/(1024**3):.2f} GB')

Q4_K_M files in repo:
  - Qwen_Qwen3-30B-A3B-Instruct-2507-Q4_K_M.gguf

downloading Qwen_Qwen3-30B-A3B-Instruct-2507-Q4_K_M.gguf...
done in 0.0 min, size 17.35 GB


## 5. 启动 llama-server（后台），等模型加载好

In [18]:
import httpx, time

server_log = '/tmp/llama-server.log'
server_proc = subprocess.Popen([
    SERVER_BIN,
    '--model', MODEL_PATH,
    '--ctx-size', str(4096 * N_PARALLEL),  # 各 slot 共享 KV 池，总池 = ctx × parallel
    '--parallel', str(N_PARALLEL),
    '--cont-batching',
    '--n-gpu-layers', '999',
    '--host', '127.0.0.1',
    '--port', str(SERVER_PORT),
    '--threads', '4',
], stdout=open(server_log, 'w'), stderr=subprocess.STDOUT)
print(f'server PID {server_proc.pid}, log → {server_log}')

# 等 /health 返回 200
url = f'http://127.0.0.1:{SERVER_PORT}'
for i in range(180):
    if server_proc.poll() is not None:
        print('SERVER CRASHED, tail of log:')
        !tail -40 {server_log}
        raise RuntimeError('server exited')
    try:
        r = httpx.get(f'{url}/health', timeout=2)
        if r.status_code == 200:
            print(f'\nserver ready after {i*2}s')
            break
    except Exception:
        pass
    if i % 10 == 0: print(f'  waiting... {i*2}s')
    time.sleep(2)
else:
    raise RuntimeError('server did not become ready in 6 min')

server PID 51315, log → /tmp/llama-server.log
  waiting... 0s

server ready after 8s


## 6. 测试一次单 prompt

In [19]:
r = httpx.post(f'{url}/v1/completions', json={
    'prompt': '本朝六卿之设虽祖周官而六部之名实沿唐制但唐之六部为尚书省之属曹\n有标点：',
    'max_tokens': 100, 'temperature': 0.0, 'stop': ['\n\n']
}, timeout=120)
print(r.json()['choices'][0]['text'][:200])

 本朝六卿之设，虽祖周官，而六部之名，实沿唐制。但唐之六部，为尚书省之属曹。  请分析该句的语法结构和翻译。


## 7. 标点逻辑：prompt + 异体字 + DP 回溯 + 长段切分 + HTTP wrapper

In [20]:
# === 与 Kaggle cell 13 同源，仅把 llm(...) 改成 HTTP POST ===
FEWSHOT_PROMPT = '''下面是给古文加中文标点的任务。

规则：
1. 保留所有原字，只在字间插入标点（，。：；？！、《》「」）。
2. 段落必须以句末符号结尾（。 ？ ！ 」 』 ）之一）；不要以 ， ： 、 ； 收尾。
3. 不要连续两个标点（除了 。」 ？」 ！」 这类引号闭合）。例如「，：」「，，」「。，」都是错的。
4. 「」必须成对出现，《》必须成对出现。开了引号一定要在合适位置闭合。
5. 句意为重，宁可让句子稍长一点，也不要把名字、官职、书名硬切成多段。

无标点：自古帝王之有天下其言行政治必有史臣纪载以垂鉴戒此古今之盛典朝廷之先务也
有标点：自古帝王之有天下，其言行政治，必有史臣纪载，以垂鉴戒，此古今之盛典，朝廷之先务也。

无标点：奉天门常朝御座后内官持一小扇金黄绢以裹之尝闻一老将军云非扇也其名卓影辟邪永乐间外国所进
有标点：奉天门常朝，御座后内官持一小扇，金黄绢以裹之。尝闻一老将军云：「非扇也，其名卓影辟邪，永乐间外国所进。」

无标点：本朝六卿之设虽祖周官而六部之名实沿唐制但唐之六部为尚书省之属曹
有标点：本朝六卿之设，虽祖周官，而六部之名实沿唐制。但唐之六部为尚书省之属曹。

无标点：吾乡布衣沈先生名璵字孟温洪武中其家坐累谪戍云南之金齒宣徳初归省坟墓乡人以其经学该愽留教子弟
有标点：吾乡布衣沈先生，名璵，字孟温。洪武中，其家坐累，谪戍云南之金齒。宣徳初，归省坟墓，乡人以其经学该愽，留教子弟。

无标点：{raw}
有标点：'''

PUNCT_SET = set('，。：；？！「」、,.:;?!"《》〈〉（）()—…——·・•‧『』〔〕\\/／')
WS_SET = set(' 　\t\n\xa0\r')

def strip_punct(s):
    return ''.join(c for c in s if c not in PUNCT_SET and c not in WS_SET)

VARIANT_MAP = {
    '衘':'衔','濓':'濂','兊':'兑','塜':'冢','贠':'员','冺':'泯','滛':'淫',
    '髠':'髡','兾':'冀','蝡':'蠕','畨':'番','桞':'柳','秪':'祇','歘':'欻',
}

from opencc import OpenCC
import heapq, functools
try:
    import yitizi; HAS_YITIZI = True
except ImportError:
    HAS_YITIZI = False
_t2s = OpenCC('t2s')

def is_pua(c): return 0xE000 <= ord(c) <= 0xF8FF

_REV_VARIANT_MAP = {}
for _k, _v in VARIANT_MAP.items():
    _REV_VARIANT_MAP.setdefault(_v, set()).add(_k)

@functools.lru_cache(maxsize=50000)
def _variants(c):
    if is_pua(c): return frozenset()
    cands = {c}
    for _ in range(4):
        new = set(cands)
        for x in cands:
            if HAS_YITIZI:
                try: new |= set(yitizi.get(x) or [])
                except: pass
            new.add(VARIANT_MAP.get(x, x))
            new |= _REV_VARIANT_MAP.get(x, set())
            new.add(_t2s.convert(x))
        if new == cands: break
        cands = new
    return frozenset(cands)

def char_eq(rc, oc):
    if is_pua(rc): return True
    if rc == oc: return True
    a = _variants(rc)
    if not a: return False
    return oc in a or bool(a & _variants(oc))

def merge_strict(raw, out):
    raw_c = [c for c in raw if c not in PUNCT_SET and c not in WS_SET]
    ri = 0; result = []
    for oc in out:
        if oc in PUNCT_SET or oc in WS_SET: result.append(oc); continue
        if ri >= len(raw_c): return None
        if char_eq(raw_c[ri], oc): result.append(raw_c[ri]); ri += 1
        else: return None
    if ri != len(raw_c): return None
    return ''.join(result)

def merge_by_count(raw, out):
    raw_c = [c for c in raw if c not in PUNCT_SET and c not in WS_SET]
    out_c = [c for c in out if c not in PUNCT_SET and c not in WS_SET]
    if len(raw_c) != len(out_c): return None
    ri = 0; result = []
    for oc in out:
        if oc in PUNCT_SET or oc in WS_SET: result.append(oc)
        else: result.append(raw_c[ri]); ri += 1
    return ''.join(result)

def align_dp(raw, out, max_cost):
    R, M = len(raw), len(out)
    INF = max_cost + 1
    dist = {(0, 0): 0}
    parents = {(0, 0): None}
    pq = [(0, 0, 0)]
    while pq:
        c, i, j = heapq.heappop(pq)
        if c > dist.get((i, j), INF): continue
        if i == R and j == M: break
        def relax(ni, nj, dc, content):
            nc = c + dc
            if nc > max_cost: return
            if nc < dist.get((ni, nj), INF):
                dist[(ni, nj)] = nc
                parents[(ni, nj)] = (i, j, content)
                heapq.heappush(pq, (nc, ni, nj))
        if j < M and (out[j] in PUNCT_SET or out[j] in WS_SET):
            relax(i, j+1, 0, out[j])
        if (i < R and j < M and not is_pua(raw[i])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET
                and char_eq(raw[i], out[j])):
            relax(i+1, j+1, 0, raw[i])
        if i < R and is_pua(raw[i]):
            relax(i+1, j, 0, '')
        if (i < R and j < M and is_pua(raw[i])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET):
            relax(i+1, j+1, 0, out[j])
        if (i+1 < R and j < M and is_pua(raw[i]) and not is_pua(raw[i+1])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET):
            relax(i+2, j+1, 0, raw[i+1])
        if (i < R and j < M and not is_pua(raw[i])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET):
            relax(i+1, j+1, 1, raw[i])
        if i < R and not is_pua(raw[i]):
            relax(i+1, j, 1, raw[i])
        if j < M and out[j] not in PUNCT_SET and out[j] not in WS_SET:
            relax(i, j+1, 1, '')
    if (R, M) not in dist: return None
    path = []
    cur = (R, M)
    while parents.get(cur) is not None:
        pi, pj, content = parents[cur]
        path.append(content)
        cur = (pi, pj)
    path.reverse()
    return ''.join(path)

def truncate_to_raw(raw, out):
    raw_c = [c for c in raw if c not in PUNCT_SET and c not in WS_SET]
    if not raw_c: return out
    ri = 0; last_j = -1
    for j, oc in enumerate(out):
        if oc in PUNCT_SET or oc in WS_SET: continue
        if ri >= len(raw_c): break
        last_j = j; ri += 1
    if ri < len(raw_c): return out
    cut = last_j + 1
    while cut < len(out) and (out[cut] in PUNCT_SET or out[cut] in WS_SET): cut += 1
    return out[:cut]

def find_start_in_out(raw, out, window=12, min_ratio=0.7):
    raw_c = [c for c in raw if c not in PUNCT_SET and c not in WS_SET][:window]
    if len(raw_c) < window: window = len(raw_c)
    out_pos = [(j, c) for j, c in enumerate(out) if c not in PUNCT_SET and c not in WS_SET]
    if len(out_pos) < window: return 0
    thresh = int(window * min_ratio)
    for s in range(len(out_pos) - window + 1):
        ok = sum(1 for k in range(window)
                 if is_pua(raw_c[k]) or char_eq(raw_c[k], out_pos[s+k][1]))
        if ok >= thresh: return out_pos[s][0]
    return 0

def rescue_cascade(raw, out):
    r = merge_strict(raw, out)
    if r is not None and ''.join(c for c in r if c not in PUNCT_SET and c not in WS_SET and not is_pua(c)) == \
       ''.join(c for c in raw if c not in PUNCT_SET and c not in WS_SET and not is_pua(c)):
        return r, 'strict'
    r = merge_by_count(raw, out)
    if r is not None: return r, 'count'
    for mc in [0, 1, 2, 3, 5, 8, 12, 15]:
        r = align_dp(raw, out, max_cost=mc)
        if r is not None: return r, f'dp_c{mc}'
    tr = truncate_to_raw(raw, out)
    if tr != out:
        r = align_dp(raw, tr, max_cost=15)
        if r is not None: return r, 'truncraw_dp'
    start = find_start_in_out(raw, out)
    if start > 0:
        sliced = out[start:]
        r = align_dp(raw, sliced, max_cost=15)
        if r is not None: return r, 'startshift_dp'
    return None, None

STOP_STRINGS = ['\n\n无标点：', '\n无标点：', '<|im_end|>', '<|endoftext|>',
                '本朝六卿之设', '自古帝王之有天下', '奉天门常朝', '吾乡布衣沈先生']

# HTTP client (thread-safe)
http_client = httpx.Client(base_url=f'http://127.0.0.1:{SERVER_PORT}', timeout=600)

def llm_http(prompt, max_tokens, temperature=0.0, stop=None):
    """线程安全的 HTTP 调用，封装成 Kaggle 版 llm(...) 的同样接口。"""
    r = http_client.post('/v1/completions', json={
        'prompt': prompt,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stop': stop or [],
        'cache_prompt': True,  # llama.cpp 自带 prompt prefix 缓存
    })
    r.raise_for_status()
    return r.json()

def punct_one_attempt(raw, jitter=0.0, max_tokens_override=None):
    prompt = FEWSHOT_PROMPT.format(raw=raw)
    max_tok = max_tokens_override if max_tokens_override else max(256, min(2000, int(len(raw) * 1.5 + 50)))
    r = llm_http(prompt, max_tokens=max_tok, temperature=jitter, stop=STOP_STRINGS)
    return r['choices'][0]['text'].strip()

# 长段切分（同 Kaggle 版）
END_PARTICLES = set('也矣焉哉')
START_WORDS = set('又初后然按')
QUOTE_OPENERS = set('曰云')

def _has_open_quote_context(raw, pos, look_back=50):
    lo = max(0, pos - look_back)
    for i in range(pos - 1, lo - 1, -1):
        if raw[i] in QUOTE_OPENERS:
            for j in range(i + 1, pos):
                if raw[j] in END_PARTICLES:
                    return False
            return True
    return False

SPLIT_QUERY_PROMPT = '''从以下 {n} 个候选位置中选最适合作为句号的位置（古文断句习惯，| 表示候选切分点）：
{candidates}

答案（只输出 {letters} 中的一个字母）：'''

def find_best_split(raw, target_pos, window=120):
    start = max(20, target_pos - window)
    end = min(len(raw) - 20, target_pos + window)
    valid = []
    for i in range(start, end):
        if i < 1 or i >= len(raw) - 1: continue
        if raw[i-1] in END_PARTICLES and raw[i] in START_WORDS:
            if not _has_open_quote_context(raw, i):
                valid.append(i)
    if not valid: return -1
    if len(valid) == 1: return valid[0]
    valid.sort(key=lambda x: abs(x - target_pos))
    candidates = sorted(valid[:3])
    if len(candidates) == 1: return candidates[0]
    letters = 'ABC'[:len(candidates)]
    options = []
    for i, pos in enumerate(candidates):
        before = raw[max(0, pos-10):pos]
        after = raw[pos:pos+10]
        options.append(f'{letters[i]}. ...{before} | {after}...')
    prompt = SPLIT_QUERY_PROMPT.format(
        n=len(candidates), candidates='\n'.join(options), letters='/'.join(letters))
    try:
        r = llm_http(prompt, max_tokens=3, temperature=0.0, stop=['\n', '。', ' ', '\t'])
        answer = r['choices'][0]['text'].strip().upper()
        for i, c in enumerate(letters):
            if c in answer: return candidates[i]
    except Exception:
        pass
    return candidates[0]

def split_long_raw(raw, max_chunk=500):
    if len(raw) <= max_chunk: return [raw]
    mid = len(raw) // 2
    split_pos = find_best_split(raw, mid)
    if split_pos < 0 or split_pos < 50 or split_pos > len(raw) - 50:
        return [raw]
    left, right = raw[:split_pos], raw[split_pos:]
    if max(len(left), len(right)) >= len(raw) - 10: return [raw]
    return split_long_raw(left, max_chunk) + split_long_raw(right, max_chunk)

def _punct_one_core(raw, max_retries=2):
    first_out_len = None
    for attempt in range(max_retries):
        if attempt == 0:
            out = punct_one_attempt(raw, jitter=0.0)
            first_out_len = len(out)
        else:
            retry_cap = max(256, int(first_out_len * 1.1)) if first_out_len else None
            out = punct_one_attempt(raw, jitter=0.3, max_tokens_override=retry_cap)
        if strip_punct(raw) == strip_punct(out):
            return out, 'strict_ok', attempt + 1
        merged, layer = rescue_cascade(raw, out)
        if merged is not None:
            return merged, layer, attempt + 1
    return None, 'gave_up', max_retries

def punct_one(raw, max_retries=2):
    if len(raw) > 500:
        chunks = split_long_raw(raw, max_chunk=500)
        if len(chunks) > 1:
            results = []
            failed = []
            for i, c in enumerate(chunks):
                out, status, tries = _punct_one_core(c, max_retries)
                if out is None:
                    results.append(c); failed.append(i)
                else:
                    results.append(out)
            merged = ''.join(results)
            imbalanced = (merged.count('「') != merged.count('」')
                          or merged.count('《') != merged.count('》')
                          or merged.count('『') != merged.count('』'))
            tags = [f'split_ok_{len(chunks)}']
            if failed: tags.append(f'partial_failed_{failed}')
            if imbalanced: tags.append('imbalanced')
            return merged, '|'.join(tags) if len(tags) > 1 else tags[0], 1
    return _punct_one_core(raw, max_retries)

# 烟雾测试
test = '朝廷每端午节赐朝官吃糕糭于午门外酒数行而出文职大臣仍从驾幸后苑观武臣射栁事毕'
out, status, tries = punct_one(test)
print(f'sample ({status}, tries={tries}):\n  {out or "(失败)"}')

sample (strict_ok, tries=1):
  朝廷每端午节，赐朝官吃糕糭于午门外，酒数行而出。文职大臣仍从驾幸后苑，观武臣射栁。事毕。


## 8. 加载数据 + checkpoint

In [21]:
import json, kagglehub

OUT_PATH = f'/content/{BOOK}{SUFFIX}-punctuated.jsonl'
FAIL_PATH = f'/content/{BOOK}{SUFFIX}-failures.jsonl'
REVIEW_PATH = f'/content/{BOOK}{SUFFIX}-needs-review.jsonl'
STATUS_PATH = f'/content/{BOOK}{SUFFIX}-status.txt'
CKPT = f'/content/{BOOK}{SUFFIX}-checkpoint.json'

ds_path = kagglehub.dataset_download('canhuilipku/mingshilu-raw')
IN_PATH = f'{ds_path}/{BOOK}.jsonl'
print(f'dataset → {IN_PATH}')

rows = [json.loads(l) for l in open(IN_PATH, encoding='utf-8') if l.strip()]
rows = [r for r in rows if r['raw'].strip()]
print(f'total rows: {len(rows)}')

# 跨步切片
my_rows = rows[PART-1::PART_OF]
print(f'part {PART}/{PART_OF}: {len(my_rows)} rows')

done_ids = set()
if os.path.exists(CKPT):
    done_ids = set(json.load(open(CKPT)))
    print(f'resuming, already done: {len(done_ids)}')
todo = [r for r in my_rows if r['id'] not in done_ids]
print(f'todo: {len(todo)}')

100%|██████████| 40.1M/40.1M [00:04<00:00, 9.76MB/s]

Extracting files...


dataset → /root/.cache/kagglehub/datasets/canhuilipku/mingshilu-raw/versions/10/mingshilu-p4-prechunked.jsonl
total rows: 21628
part 1/1: 21628 rows
todo: 21628


## 9. 并行主循环（ThreadPoolExecutor × N_PARALLEL）

In [22]:
import time, traceback, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

stats = Counter()
stats_lock = threading.Lock()
file_lock = threading.Lock()
ckpt_lock = threading.Lock()

out_f = open(OUT_PATH, 'a', encoding='utf-8')
fail_f = open(FAIL_PATH, 'a', encoding='utf-8')
review_f = open(REVIEW_PATH, 'a', encoding='utf-8')

PRINT_EVERY = 50
completed_count = 0
count_lock = threading.Lock()
t0 = time.time()

def process_one(r):
    global completed_count
    try:
        out, status, tries = punct_one(r['raw'])
    except Exception as e:
        with stats_lock: stats['err'] += 1
        return r['id'], 'err'

    if status != 'gave_up':
        rec = {'id': r['id'], 'raw': r['raw'], 'punct': out, 'source': status, 'tries': tries}
        with file_lock:
            out_f.write(json.dumps(rec, ensure_ascii=False) + '\n'); out_f.flush()
            if 'partial_failed' in status or 'imbalanced' in status:
                review_f.write(json.dumps(rec, ensure_ascii=False) + '\n'); review_f.flush()
    else:
        try:
            last = punct_one_attempt(r['raw'], jitter=0.3)
        except Exception:
            last = ''
        fail_rec = {'id': r['id'], 'raw': r['raw'], 'last_attempt': last, 'status': status}
        with file_lock:
            fail_f.write(json.dumps(fail_rec, ensure_ascii=False) + '\n'); fail_f.flush()

    with stats_lock:
        stats[status] += 1
        done_ids.add(r['id'])

    with count_lock:
        completed_count += 1
        cc = completed_count
    if cc % PRINT_EVERY == 0 or cc == len(todo):
        with ckpt_lock:
            json.dump(list(done_ids), open(CKPT, 'w'))
        elapsed = time.time() - t0
        rate = cc / elapsed
        eta_min = (len(todo) - cc) / rate / 60 if rate > 0 else 0
        with stats_lock:
            total_ok = sum(v for k, v in stats.items() if k not in ('gave_up', 'err'))
            top = ' '.join(f'{k}={v}' for k, v in stats.most_common(4))
        line = f'[{cc}/{len(todo)} {cc/len(todo)*100:.1f}% rate={rate:.2f}/s ETA={eta_min:.0f}m] ok={total_ok} | {top}'
        print(line)
        with open(STATUS_PATH, 'w') as sf: sf.write(line + '\n')

    return r['id'], status

with ThreadPoolExecutor(max_workers=N_PARALLEL) as ex:
    futures = {ex.submit(process_one, r): r for r in todo}
    for fut in as_completed(futures):
        try: fut.result()
        except Exception as e:
            print(f'task error: {e}')
            traceback.print_exc()

out_f.close(); fail_f.close(); review_f.close()
with ckpt_lock: json.dump(list(done_ids), open(CKPT, 'w'))

total = sum(stats.values())
total_ok = sum(v for k, v in stats.items() if k not in ('gave_up', 'err'))
print(f'\nDONE — {total_ok}/{total} ({total_ok/total*100:.1f}%)  total time={(time.time()-t0)/60:.1f} min')
print('layers:')
for k, v in sorted(stats.items(), key=lambda x: -x[1]):
    print(f'  {v:5d}  {k}')
print(f'\n输出: {OUT_PATH}\n失败: {FAIL_PATH}\n复查: {REVIEW_PATH}')

[50/21628 0.2% rate=1.57/s ETA=228m] ok=49 | strict_ok=27 strict=19 dp_c5=1 dp_c3=1
[100/21628 0.5% rate=1.66/s ETA=216m] ok=99 | strict_ok=48 strict=44 dp_c5=2 count=2
[150/21628 0.7% rate=1.67/s ETA=214m] ok=149 | strict_ok=69 strict=68 count=3 dp_c8=3
[200/21628 0.9% rate=1.44/s ETA=247m] ok=198 | strict_ok=87 strict=87 count=6 dp_c12=4
[250/21628 1.2% rate=1.46/s ETA=244m] ok=248 | strict_ok=111 strict=107 count=9 dp_c12=4
[300/21628 1.4% rate=1.56/s ETA=228m] ok=298 | strict_ok=138 strict=126 count=12 dp_c12=4
[350/21628 1.6% rate=1.54/s ETA=230m] ok=348 | strict_ok=163 strict=142 count=14 dp_c12=8
[400/21628 1.8% rate=1.57/s ETA=225m] ok=398 | strict_ok=190 strict=162 count=15 dp_c12=8
[450/21628 2.1% rate=1.57/s ETA=225m] ok=447 | strict_ok=215 strict=178 count=16 dp_c12=9
[500/21628 2.3% rate=1.67/s ETA=211m] ok=497 | strict_ok=240 strict=195 count=19 dp_c1=11
[550/21628 2.5% rate=1.77/s ETA=199m] ok=548 | strict_ok=270 strict=214 count=21 dp_c1=11
[600/21628 2.8% rate=1.74/s E

## 10. 打包输出 + 写到 Drive（可选）

import httpx
print(httpx.get(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=5).status_code)


In [23]:
"""跑完一次性打包 + 上传到 Kaggle dataset。

跑完后我用 `kaggle datasets download canhuilipku/mingshilu-p5-output -p ~/Downloads/` 拉到本地，
直接喂给 merge-mingshilu-retry.py 合回原段。
"""
import os, shutil, zipfile, json, subprocess, time

# 装 kaggle CLI（kagglehub 不能 upload）
subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)

# 把所有 output 文件复制到上传目录
upload_dir = '/tmp/p5-output-upload'
shutil.rmtree(upload_dir, ignore_errors=True)
os.makedirs(upload_dir, exist_ok=True)

zip_path = f'{upload_dir}/{BOOK}{SUFFIX}-output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [OUT_PATH, FAIL_PATH, REVIEW_PATH, STATUS_PATH, CKPT]:
        if os.path.exists(p): z.write(p, os.path.basename(p))
print(f'zipped → {zip_path}  size {os.path.getsize(zip_path)/1024/1024:.2f} MB')

# 同时也单独传 punctuated.jsonl（方便 merge 脚本直接消费）
for p in [OUT_PATH, FAIL_PATH, REVIEW_PATH, STATUS_PATH]:
    if os.path.exists(p):
        shutil.copy(p, upload_dir)

# 写 dataset metadata
meta = {
    'id': OUTPUT_DATASET,
    'title': f'Mingshilu {BOOK} Output',
    'licenses': [{'name': 'CC0-1.0'}],
}
json.dump(meta, open(f'{upload_dir}/dataset-metadata.json', 'w'), indent=2)

# 第一次 create，后续 version
created = False
try:
    print('try kaggle datasets create ...')
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', upload_dir],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    created = r.returncode == 0
except Exception as e:
    print(f'create exc: {e}')

if not created:
    print('\ndataset 已存在，改用 version 推新版本...')
    r = subprocess.run(['kaggle', 'datasets', 'version', '-p', upload_dir,
                        '-m', f'auto run {int(time.time())}'],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

print(f'\n✅ 输出已上传到 https://www.kaggle.com/datasets/{OUTPUT_DATASET}')
print(f'本地拉取: kaggle datasets download {OUTPUT_DATASET} -p ~/Downloads/mingshilu-p5/ --unzip')


zipped → /tmp/p5-output-upload/mingshilu-p4-prechunked-output.zip  size 4.57 MB
try kaggle datasets create ...
Starting upload for file mingshilu-p4-prechunked-punctuated.jsonl
Upload successful: mingshilu-p4-prechunked-punctuated.jsonl (13MB)
Starting upload for file mingshilu-p4-prechunked-needs-review.jsonl
Upload successful: mingshilu-p4-prechunked-needs-review.jsonl (0B)
Starting upload for file mingshilu-p4-prechunked-status.txt
Upload successful: mingshilu-p4-prechunked-status.txt (98B)
Starting upload for file mingshilu-p4-prechunked-output.zip
Upload successful: mingshilu-p4-prechunked-output.zip (5MB)
Starting upload for file mingshilu-p4-prechunked-failures.jsonl
Upload successful: mingshilu-p4-prechunked-failures.jsonl (297KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/canhuilipku/mingshilu-p4-output


  0%|          | 0.00/13.2M [00:00<?, ?B/s]
  7%|▋         | 928k/13.2M [00:00<00:09, 1.31MB/s]
  9%|▉         | 1.16M/13

## 11. 停 server（清理）

In [24]:
server_proc.terminate()
try:
    server_proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    server_proc.kill()
print('server stopped')

server stopped
